# Ulasan Aplikasi Qara'a

Notebook ini menjalankan pipeline analisis sentimen dan topik untuk ulasan aplikasi Qara'a. Kode utama berada di folder `src/`, notebook hanya memanggil fungsi dan menampilkan hasil.

In [1]:
import os
import sys
from pathlib import Path
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# agar modul di folder src/ bisa diimpor dari root proyek
sys.path.append('.')

In [2]:
# flag agar sel yang menulis ulang data tidak dijalankan tanpa sengaja
JALANKAN_SCRAPING = False

# path file, sementara masih di root. pindahkan dengan mengubah nilai di sini
PATH_RAW = 'ulasan_qaraa_raw.csv'
PATH_LABEL_AUTO = 'ulasan_qaraa_label_auto.csv'
PATH_PERLU_CEK = 'ulasan_qaraa_perlu_cek.csv'
PATH_LABEL_MANUAL = 'ulasan_qaraa_label_manual.csv'
PATH_FINAL = 'ulasan_qaraa_final.csv'
PATH_TOPIK = 'ulasan_qaraa_topik.csv'
PATH_MODEL = 'models/model_svm_terbaik.joblib'

# tampilkan semua kolom agar mudah dibaca di layar
pd.set_option('display.max_columns', None)

# pastikan file data mentah sudah ada sebelum section berikutnya dijalankan
print(f"data mentah ditemukan: {Path(PATH_RAW).exists()}")

data mentah ditemukan: True


In [3]:
from src.preprocessing import preprocess_teks
from src.inference import muat_model, prediksi

print(f"modul src berhasil diimpor")

modul src berhasil diimpor


## Scraping Data

Section ini memuat data mentah yang sudah ada. Pengambilan ulang dari Google Play dinonaktifkan secara default, karena ulasan bisa berubah dan `id` pada validasi manual bergantung pada urutan baris raw yang sekarang.

In [4]:
# konfigurasi scraping, hanya dipakai jika JALANKAN_SCRAPING = True
APP_ID = 'com.bismillah.amaljariyah'   # id aplikasi qara'a
START_DATE = '2020-01-01'              # sesuai batasan masalah
END_DATE = '2025-11-15'                # sesuai batasan masalah

In [5]:
from src.scraping import ambil_ulasan

if JALANKAN_SCRAPING:
    # simpan ke file baru agar raw yang dipakai validasi manual tidak tertimpa
    print("mengambil ulasan dari google play, mohon tunggu...")
    df_raw = ambil_ulasan(APP_ID, START_DATE, END_DATE)
    df_raw.to_csv('ulasan_qaraa_raw_baru.csv', index=False)
    print(f"data scraping disimpan ke ulasan_qaraa_raw_baru.csv: {len(df_raw)} baris")
else:
    df_raw = pd.read_csv(PATH_RAW)
    print(f"data mentah dimuat dari {PATH_RAW}: {len(df_raw)} baris")

data mentah dimuat dari ulasan_qaraa_raw.csv: 11293 baris


In [6]:
print(f"total ulasan (rentang {START_DATE} s.d. {END_DATE}): {len(df_raw)}")
print(f"jumlah kolom: {df_raw.shape[1]}")
df_raw.head()

total ulasan (rentang 2020-01-01 s.d. 2025-11-15): 11293
jumlah kolom: 11


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,c672dd3b-1f4a-4477-8a3f-3292c09f1f3c,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,maaf kak untuk notifikasi adzan nya kok engga ...,5,0,5.1.30,2025-11-02 22:25:02,"Assalamu'alaikum kak, untuk mengatasi masalah ...",2025-11-04 13:31:50,5.1.30
1,9994dcc1-3c67-437d-9bfc-11d521ade639,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Terimakasih atas aplikasi nya semoga berkemban...,5,0,NaN,2025-11-01 13:48:42,"MasyaAllah, terima kasih banyak Kak atas doa d...",2025-11-04 13:29:44,NaN
2,39866958-3b13-4f43-9416-899d7a4acf39,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,aku dah bayar kok langgananku nga aktif min,4,0,5.1.30,2025-10-30 15:36:37,Mohon maaf ya Kak atas ketidaknyamanannya 🙏\nB...,2025-11-04 13:27:09,5.1.30
3,7a19470d-adcf-4017-8a09-e52a62391c7a,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Aplikasinya sangat bagus, namun ketika terjadi...",1,0,5.1.30,2025-10-23 07:20:33,"Assalamu'alaikum kak, mohon maaf atas ketidakn...",2025-10-30 09:33:27,5.1.30
4,7384cefb-4eba-4344-801e-5a5a94fbcb25,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,buat Qara'a tetap semangat untuk mengembangkan...,5,0,NaN,2025-10-21 09:04:05,MasyaAllah.. Jazakallah khair kak atas penilai...,2025-11-04 13:14:22,NaN


## Preprocessing Text

Setiap  tahap dijalankan berurutan pada seluruh data. Hasil tiap tahap bisa ditulis ke file interim jika flag `JALANKAN_PIPELINE` diaktifkan.

In [7]:
# path file interim dan log, hanya dipakai jika TULIS_INTERIM = True
PATH_TAHAP = {
    'casefolding': 'ulasan_qaraa_casefolding.csv',
    'cleaning': 'ulasan_qaraa_cleaning.csv',
    'normalisasi': 'ulasan_qaraa_normalisasi.csv',
    'stopword': 'ulasan_qaraa_stopword.csv',
    'stemming': 'ulasan_qaraa_stemming.csv',
    'tokenizing': 'ulasan_qaraa_tokenizing.csv',
}
PATH_LOG_DROP = 'log_drop_ulasan.csv'

# ganti jadi True hanya jika memang ingin menulis ulang interim.
# setelah itu label_auto, label_manual, perlu_cek, dan final wajib dibuat ulang.
TULIS_INTERIM = False

In [8]:
from src.preprocessing import jalankan_tahap, simpan_tahap

tahap, log_drop_tahap = jalankan_tahap(df_raw)

print(f"data setelah tokenizing: {len(tahap['tokenizing'])} baris")
print(f"total data dibuang: {len(log_drop_tahap)}")
print(log_drop_tahap['tahap'].value_counts())

if TULIS_INTERIM:
    simpan_tahap(tahap, log_drop_tahap, PATH_TAHAP, PATH_LOG_DROP)
    print("peringatan: file interim ditulis ulang. buat ulang label_auto, label_manual, perlu_cek, dan final.")
else:
    print("penulisan file dinonaktifkan (TULIS_INTERIM = False)")

data setelah tokenizing: 11044 baris
total data dibuang: 249
tahap
cleaning    245
stopword      4
Name: count, dtype: int64
penulisan file dinonaktifkan (TULIS_INTERIM = False)


In [9]:
final_cek = pd.read_csv(PATH_FINAL).set_index('id')['content_final'].fillna('')
stem = tahap['stemming'].set_index('id')['content_stem']

cocok = stem.loc[final_cek.index] == final_cek
print(f"cocok dengan content_final: {cocok.sum()}/{len(final_cek)} (harus 11044/11044)")

hasil_validasi = tahap['stemming']['content_stem'].str.contains(r'\baji\b').sum()
print(f"validasi 'aji' (harus 0): {hasil_validasi}")

if TULIS_INTERIM:
    print("peringatan: TULIS_INTERIM aktif, hasil verifikasi ini belum sinkron dengan label.")

cocok dengan content_final: 11044/11044 (harus 11044/11044)
validasi 'aji' (harus 0): 0


## Pelabelan Data

Label otomatis dibuat dari skor, lalu kandidat validasi manual ditandai aturan. Label manual digabung berdasarkan `id`, lalu data final dibangun ulang dari teks mentah.

In [10]:
PATH_LOG_VALIDASI = 'log_drop_validasi.csv'
PATH_LOG_FINAL = 'log_drop_final.csv'

# tulis label_auto, label_manual, log validasi, dan final. aktifkan hanya jika memang perlu menulis ulang
TULIS_LABEL = False

# tulis perlu cek hanya jika file belum ada, karena file ini berisi pengisian manual
TULIS_PERLU_CEK = False

In [11]:
from src.labeling import buat_label_auto

df_label = buat_label_auto(tahap['tokenizing'])
print(f"label otomatis di memori: {len(df_label)} baris")
print(df_label['label'].value_counts())

# bandingkan dengan label_auto yang dipakai untuk validasi manual, harus identik
ref_auto = pd.read_csv(PATH_LABEL_AUTO)
gab_auto = df_label.merge(ref_auto[['id', 'label']], on='id', how='outer',
                          suffixes=('', '_ref'), indicator=True)
print("id hanya di memori:", (gab_auto['_merge'] == 'left_only').sum(), "(harus 0)")
print("id hanya di file:", (gab_auto['_merge'] == 'right_only').sum(), "(harus 0)")
print("label berbeda:", (gab_auto['label'] != gab_auto['label_ref']).sum(), "(harus 0)")

if TULIS_LABEL:
    df_label.to_csv(PATH_LABEL_AUTO, index=False, encoding='utf-8-sig')
    print("label_auto ditulis ulang. buat ulang validasi manual dan data final.")

label otomatis di memori: 11044 baris
label
positif    9962
negatif     654
netral      428
Name: count, dtype: int64
id hanya di memori: 0 (harus 0)
id hanya di file: 0 (harus 0)
label berbeda: 0 (harus 0)


In [12]:
from pathlib import Path
from src.labeling import buat_perlu_cek

df_perlu_baru = buat_perlu_cek(df_label)
print(f"kandidat perlu cek di memori: {len(df_perlu_baru)}")
print(df_perlu_baru['alasan'].str.split('; ').explode().value_counts())

# bandingkan dengan file yang sudah berisi pengisian manual
ref_perlu = pd.read_csv(PATH_PERLU_CEK, dtype={'label_manual_baru': str})
print("id sama dengan file:", set(df_perlu_baru['id']) == set(ref_perlu['id']))
print("baris terisi di file:", ref_perlu['label_manual_baru'].notna().sum())

if TULIS_PERLU_CEK:
    if Path(PATH_PERLU_CEK).exists():
        print("file perlu cek sudah ada, tidak ditimpa")
    else:
        df_perlu_baru.to_csv(PATH_PERLU_CEK, index=False, encoding='utf-8-sig')
        print("file perlu cek dibuat")

kandidat perlu cek di memori: 5101
alasan
pendek                      4381
netral                       428
negatif_fitur_premium        242
negatif_ada_kata_positif     105
positif_ada_kata_negatif     101
Name: count, dtype: int64
id sama dengan file: False
baris terisi di file: 703


In [13]:
from src.labeling import gabung_label_manual, validasi_label

df_auto = pd.read_csv(PATH_LABEL_AUTO)
df_perlu = pd.read_csv(PATH_PERLU_CEK, dtype={'label_manual_baru': str})

df_gabung, log_validasi, berubah = gabung_label_manual(df_auto, df_perlu)
validasi_label(df_gabung)

print(f"baris setelah gabung: {len(df_gabung)}, dihapus: {len(log_validasi)}")
print(f"perubahan label: {berubah}")

if TULIS_LABEL:
    df_gabung.to_csv(PATH_LABEL_MANUAL, index=False, encoding='utf-8-sig')
    log_validasi.to_csv(PATH_LOG_VALIDASI, index=False, encoding='utf-8-sig')
    print("label manual dan log validasi ditulis")

baris setelah gabung: 11044, dihapus: 0
perubahan label: 699


In [14]:
from src.labeling import bangun_final

df_final, log_final = bangun_final(df_gabung, df_raw)
print(f"dibuang saat finalisasi: {len(log_final)}")
print(f"jumlah data final: {len(df_final)}")
print(df_final['label'].value_counts())

# bandingkan dengan final yang sudah dipakai LDA dan SVM, harus identik
ref_final = pd.read_csv(PATH_FINAL).set_index('id')
cur = df_final.set_index('id')
ref_aligned = ref_final.reindex(cur.index)
print("id sama dengan file:", set(cur.index) == set(ref_final.index))
print("content_final sama:", (cur['content_final'] == ref_aligned['content_final'].fillna('')).sum(), f"dari {len(cur)}")
print("label sama:", (cur['label'] == ref_aligned['label']).sum(), f"dari {len(cur)}")

if TULIS_LABEL:
    log_final.to_csv(PATH_LOG_FINAL, index=False, encoding='utf-8-sig')
    df_final.to_csv(PATH_FINAL, index=False, encoding='utf-8-sig')
    print("data final ditulis ulang")

dibuang saat finalisasi: 0
jumlah data final: 11044
label
positif    9742
negatif    1020
netral      282
Name: count, dtype: int64
id sama dengan file: True
content_final sama: 11044 dari 11044
label sama: 11044 dari 11044


## Visualisasi Sentimen

### Distribusi Sentimen Keseluruhan

In [ ]:
# hitung jumlah ulasan per label sentimen
sentimen_counts = df['label'].value_counts()

# tentukan warna tetap agar setiap label konsisten di semua grafik
warna_map = {'positif': '#2ecc71', 'negatif': '#e74c3c', 'netral': '#95a5a6'}
colors = [warna_map[label] for label in sentimen_counts.index]

plt.figure(figsize=(8, 5))
bars = plt.bar(sentimen_counts.index, sentimen_counts.values, color=colors)

# tampilkan jumlah ulasan di atas setiap batang
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval + 20, int(yval),
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.title("Distribusi Sentimen Ulasan Aplikasi Qara'a", fontsize=14)
plt.ylabel('Jumlah Ulasan')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

### Proporsi Sentimen

In [ ]:
# pie chart proporsi sentimen dengan warna yang sama seperti bar chart
plt.figure(figsize=(6, 6))
plt.pie(sentimen_counts, labels=sentimen_counts.index, autopct='%1.1f%%',
        startangle=90, colors=colors)
plt.title('Persentase Proporsi Sentimen')
plt.show()

### Distribusi Sentimen per Fitur

In [7]:
# kata kunci tiap fitur dicocokkan dengan batas kata atau regex agar tidak mengenai kata lain, misal "belajar" tidak mengenai "belajarnya"
fitur_dict = {
    'Materi Belajar': r'\b(?:belajar|materi)\b',
    "Baca Al-Qur'an": r'\b(?:membaca|baca|alquran|mengaji)\b',
    'Sensor Suara': r'\b(?:suara|rekam|perekaman)\b',
    'Notifikasi Adzan': r'\b(?:adzan|azan|notifikasi|alarm)\b',
    'Layanan Premium': r'\b(?:premium|langganan|membayar|bayar|iklan)\b',
}

kolom_teks = 'content_final'

In [8]:
# hitung jumlah sentimen pada ulasan yang mengandung kata kunci tiap fitur
data_fitur = []

for fitur, keywords in fitur_dict.items():
    df_fitur = df[df[kolom_teks].str.contains(keywords, case=False, na=False)]
    counts = df_fitur['label'].value_counts().to_dict()

    data_fitur.append({
        'Fitur': fitur,
        'positif': counts.get('positif', 0),
        'negatif': counts.get('negatif', 0),
        'netral': counts.get('netral', 0)
    })

# ubah hasil perhitungan menjadi dataframe untuk grafik
df_grafik = pd.DataFrame(data_fitur)

In [ ]:
# warna urut sesuai kolom positif, negatif, netral
warna_sentimen = ['#2ecc71', '#e74c3c', '#95a5a6']

ax = df_grafik.set_index('Fitur').plot(
    kind='bar',
    stacked=False,
    figsize=(12, 7),
    color=warna_sentimen,
    width=0.7,
    edgecolor='black',
    linewidth=0.5
)

# atur judul, label sumbu, dan legenda agar rapi
plt.title("Distribusi Sentimen Berdasarkan Fitur Aplikasi Qara'a", fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Fitur Aplikasi', fontsize=13, fontweight='bold', labelpad=10)
plt.ylabel('Jumlah Ulasan', fontsize=13, fontweight='bold', labelpad=10)
plt.xticks(rotation=0, fontsize=12)
plt.yticks(fontsize=12)
plt.legend(title='Kategori Sentimen', title_fontsize=12, fontsize=11, loc='upper right')

# tampilkan angka di atas batang yang nilainya lebih dari 0
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom',
                    fontsize=10, color='black', xytext=(0, 3),
                    textcoords='offset points')

plt.tight_layout()

## Topic Modeling LDA

In [ ]:
# sumber teks lda, hanya memakai file final (sudah divalidasi manual)
FILE_SUMBER = 'ulasan_qaraa_final.csv'
KOLOM_TEKS = 'content_final'

def ke_token(nilai):
    # content_final berupa kalimat yang sudah dipisah spasi
    return str(nilai).split()

df_lda_all = pd.read_csv(FILE_SUMBER)
df_lda_all[KOLOM_TEKS] = df_lda_all[KOLOM_TEKS].fillna('').astype(str)
print(f"data lda dimuat dari {FILE_SUMBER}: {len(df_lda_all)} baris")

In [18]:
# kata fungsi khusus lda, terpisah dari stopword utama untuk svm
STOP_LDA = {'sudah', 'tapi', 'mau', 'kalau', 'oke', 'baru', 'aja', 'saja', 'gak', 'nya', "kan", "kalo"}

# pujian, ungkapan religius, dan sapaan tanpa aspek
# dibuang setelah bigram terbentuk, supaya entri berbentuk frasa (misal assalamu_alaikum) ikut terfilter
STOP_EVAL = {
    'bagus', 'baik', 'sangat', 'sekali', 'mantap', 'keren',
    'semoga', 'amin', 'aamiin', 'alhamdulillah', 'allah', 'masyaallah',
    'barakallah', 'jazakallah', 'subhanallah', 'berkah', 'amal',
    'terimakasih', 'terima', 'kasih', 'aplikasi', 'qaraa',
    'selalu', 'makin', 'semua', 'buat', 'jadi', 'banyak', 'lebih',
    'insyaallah', 'taala', 'mudahan', 'swt', 'jariyah',
    'assalamualaikum', 'mohon_maaf', 'mohon_bantu', 'bagaimana_cara', 'apa_apa',
    # frasa hasil bigram yang muncul di output terakhir
    'assalamu_alaikum', 'maa_syaa', 'allah_swt', 'jadi_amal',
    'masyaallah_tabarakallah', 'kasih_bintang', 'mohon_baik',
    'jazakumullah_khair', 'tim_qaraa',
}

# kata yang tidak ditampilkan di word cloud (negasi "tidak" sengaja tidak difilter)
STOP_WC = {'sangat', 'jadi', 'buat', 'aja', 'saja', 'lebih', 'banyak', 'nya', 'dulu'}

# parameter per sentimen: ambang kata jarang dan parameter bigram
PARAM = {
    'positif': {'no_below': 5, 'min_count': 5, 'threshold': 10},
    'negatif': {'no_below': 3, 'min_count': 3, 'threshold': 5},
}

SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
SEED_FINAL = 42
RENTANG_TOPIK = range(2, 9)
PASSES_EVAL = 10
PASSES_FINAL = 50

In [12]:
def siapkan_teks(df_sub):
    # tahap awal: hanya buang kata fungsi lda, token belum dibentuk frasa
    ids = df_sub['id'].tolist()
    texts = [[w for w in ke_token(t) if w not in STOP_LDA] for t in df_sub[KOLOM_TEKS]]
    return ids, texts

def buat_frasa(texts, min_count, threshold):
    # bigram dari data, misal tidak_buka dan langganan_dulu
    phraser = Phraser(Phrases(texts, min_count=min_count, threshold=threshold, delimiter='_'))
    return [phraser[t] for t in texts]

def filter_eval(ids, texts):
    # buang kata evaluatif dan frasa religius setelah bigram terbentuk, lalu buang dokumen kosong
    pasangan = [(i, [w for w in t if w not in STOP_EVAL]) for i, t in zip(ids, texts)]
    pasangan = [(i, t) for i, t in pasangan if len(t) > 0]
    return [i for i, _ in pasangan], [t for _, t in pasangan]

def contoh_frasa(texts, n=15):
    # hitung dokumen per frasa, dipanggil setelah filter agar yang tampil adalah frasa yang benar-benar dipakai
    hitung = Counter(w for t in texts for w in set(t) if '_' in w)
    print(f"jumlah frasa terbentuk: {len(hitung)}")
    print(hitung.most_common(n))

def evaluasi_lda(texts, nama, no_below):
    dictionary = corpora.Dictionary(texts)
    dictionary.filter_extremes(no_below=no_below, no_above=0.5)
    corpus = [dictionary.doc2bow(t) for t in texts]

    # hitung c_v untuk setiap kombinasi k dan seed
    hasil = []
    for k in RENTANG_TOPIK:
        for seed in SEEDS:
            model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=k,
                             random_state=seed, passes=PASSES_EVAL, alpha='auto', eta='auto')
            cv = CoherenceModel(model=model, texts=texts, dictionary=dictionary,
                                coherence='c_v').get_coherence()
            hasil.append({'k': k, 'seed': seed, 'c_v': cv})

    df_h = pd.DataFrame(hasil)
    ringkasan = df_h.groupby('k')['c_v'].agg(['mean', 'std']).reset_index()

    print(f"\n[{nama}] ringkasan c_v (rata-rata dan simpangan baku dari {len(SEEDS)} seed):")
    print(ringkasan.round(4).to_string(index=False))
    print(f"\n[{nama}] c_v per seed:")
    print(df_h.pivot(index='k', columns='seed', values='c_v').round(3).to_string())

    k = pilih_k_satu_sd(ringkasan, nama)
    print(f"[{nama}] ukuran kosakata: {len(dictionary)}")
    return dictionary, corpus, df_h, ringkasan, k

def pilih_k_satu_sd(ringkasan, nama):
    # ambil k terkecil yang rata-ratanya masih dalam satu simpangan baku dari nilai tertinggi
    best = ringkasan.loc[ringkasan['mean'].idxmax()]
    batas = best['mean'] - best['std']
    k = int(ringkasan.loc[ringkasan['mean'] >= batas, 'k'].min())
    print(f"[{nama}] c_v tertinggi di k={int(best['k'])}, batas satu simpangan baku: {batas:.4f}, k terpilih: {k}")
    return k

def latih_final(dictionary, corpus, k, nama):
    # model final dengan seed tetap agar hasil bisa direproduksi
    model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=k,
                     random_state=SEED_FINAL, passes=PASSES_FINAL, alpha='auto', eta='auto')

    print(f"\n[{nama}] topik (15 kata teratas):")
    for idx, topik in model.show_topics(num_topics=-1, num_words=15, formatted=False):
        print(f"  topik {idx}: " + ', '.join(w for w, _ in topik))
    return model

def jalankan_sentimen(nama):
    p = PARAM[nama]
    df_sub = df_lda_all[df_lda_all['label'] == nama]

    # urutan: stopword lda -> bigram -> stoplist evaluatif
    ids, texts = siapkan_teks(df_sub)
    texts = buat_frasa(texts, p['min_count'], p['threshold'])
    ids, texts = filter_eval(ids, texts)
    print(f"[{nama}] dokumen setelah stoplist: {len(texts)} (dibuang {len(df_sub) - len(texts)})")
    contoh_frasa(texts)

    dictionary, corpus, df_h, ringkasan, k = evaluasi_lda(texts, nama, p['no_below'])
    model = latih_final(dictionary, corpus, k, nama)

    # topik dominan tiap ulasan, disimpan dengan id agar bisa digabung kembali
    out = []
    for id_, bow in zip(ids, corpus):
        t, pr = max(model.get_document_topics(bow, minimum_probability=0.0), key=lambda x: x[1])
        out.append((id_, t, round(pr, 4)))
    df_out = pd.DataFrame(out, columns=['id', 'topik_dominan', 'probabilitas'])
    df_out['sentimen'] = nama

    return {'model': model, 'df_h': df_h, 'ringkasan': ringkasan, 'k': k, 'out': df_out}

In [ ]:
hasil_lda = {}
hasil_lda['positif'] = jalankan_sentimen('positif')

In [ ]:
hasil_lda['negatif'] = jalankan_sentimen('negatif')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, nama in zip(axes, ['positif', 'negatif']):
    r = hasil_lda[nama]['ringkasan']
    k = hasil_lda[nama]['k']
    ax.plot(r['k'], r['mean'], marker='o', label='rata-rata c_v')
    ax.fill_between(r['k'], r['mean'] - r['std'], r['mean'] + r['std'], alpha=0.2)
    ax.axvline(k, color='red', linestyle='--', label=f'terpilih k={k}')
    ax.set_title(f'Koherensi c_v ({len(SEEDS)} seed): {nama}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Jumlah Topik')
    ax.set_ylabel('c_v')
    ax.grid(linestyle='--', alpha=0.5)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
df_topik = pd.concat([hasil_lda['positif']['out'], hasil_lda['negatif']['out']], ignore_index=True)
df_topik.to_csv('ulasan_qaraa_topik.csv', index=False)
print("topik per sentimen disimpan ke: ulasan_qaraa_topik.csv")

for nama in ['positif', 'negatif']:
    proporsi = hasil_lda[nama]['out']['topik_dominan'].value_counts(normalize=True).sort_index().round(3)
    print(f"[{nama}] proporsi topik dominan:\n{proporsi}\n")

In [ ]:
def gambar_wordcloud(model, nama, colormap):
    for idx in range(model.num_topics):
        bobot = {w: float(p) for w, p in model.show_topic(idx, topn=50) if w not in STOP_WC}
        wc = WordCloud(width=800, height=500, background_color='white',
                       colormap=colormap).generate_from_frequencies(bobot)

        plt.figure(figsize=(10, 6))
        plt.imshow(wc, interpolation='bilinear')
        plt.title(f'Word Cloud {nama} - Topik {idx}', fontsize=14, fontweight='bold')
        plt.axis('off')
        plt.show()

gambar_wordcloud(hasil_lda['positif']['model'], 'Positif', 'Greens')
gambar_wordcloud(hasil_lda['negatif']['model'], 'Negatif', 'Reds')

## Pemodelan SVM

In [ ]:
df = pd.read_csv('ulasan_qaraa_final.csv')
df['content_final'] = df['content_final'].fillna('').astype(str)

X = df['content_final']
y = df['label']
groups = df['content_final']   # teks yang sama dianggap satu grup

# satu lipatan dipakai sebagai data uji, teks yang sama tidak pernah terbagi antara latih dan uji
sgkf_uji = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
idx_train, idx_test = next(sgkf_uji.split(X, y, groups=groups))

X_train, X_test = X.iloc[idx_train], X.iloc[idx_test]
y_train, y_test = y.iloc[idx_train], y.iloc[idx_test]
g_train = groups.iloc[idx_train]

print(f"jumlah data: {len(df)}, latih: {len(X_train)}, uji: {len(X_test)}")
print(f"teks yang sama di latih dan uji: {len(set(X_train) & set(X_test))} (harus 0)")
print("proporsi label latih:")
print(y_train.value_counts(normalize=True).round(3))
print("jumlah label uji:")
print(y_test.value_counts())

In [3]:
# lipatan cv di data latih juga berbasis grup, disimpan sebagai daftar agar konsisten di semua pemanggilan
splits_latih = list(
    StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    .split(X_train, y_train, groups=g_train)
)

RENTANG_C = [0.01, 0.1, 0.5, 1, 5, 10]
SKEMA = ['tanpa_penanganan', 'class_weight', 'smote']
NAMA_SKEMA = {
    'tanpa_penanganan': 'Tanpa penanganan',
    'class_weight': 'Class weight',
    'smote': 'SMOTE',
}
LABELS = ['positif', 'negatif', 'netral']

def buat_tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, sublinear_tf=True)

def buat_svm(skema, C):
    # class weight hanya aktif pada skema class_weight
    bobot = 'balanced' if skema == 'class_weight' else None
    return LinearSVC(C=C, class_weight=bobot, max_iter=5000, random_state=42)

def bangun_pipeline(skema, C, kalibrasi=False):
    svm = buat_svm(skema, C)
    # kalibrasi menghasilkan probabilitas, hanya dipakai untuk model inference
    if kalibrasi:
        svm = CalibratedClassifierCV(svm, cv=splits_latih, method='sigmoid')
    # imblearn pipeline memastikan smote hanya berjalan pada data latih tiap fold
    if skema == 'smote':
        return ImbPipeline([
            ('tfidf', buat_tfidf()),
            ('smote', SMOTE(random_state=42)),
            ('svm', svm),
        ])
    return SkPipeline([
        ('tfidf', buat_tfidf()),
        ('svm', svm),
    ])

In [ ]:
hasil_cv = []
for skema in SKEMA:
    for C in RENTANG_C:
        skor = cross_val_score(
            bangun_pipeline(skema, C), X_train, y_train,
            cv=splits_latih, scoring='f1_macro', n_jobs=-1,
        )
        hasil_cv.append({'skema': skema, 'C': C, 'mean': skor.mean(), 'std': skor.std()})
        print(f"[{NAMA_SKEMA[skema]}] C={C}: macro f1 = {skor.mean():.4f} ± {skor.std():.4f}")

df_cv = pd.DataFrame(hasil_cv)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

for skema, grup in df_cv.groupby('skema'):
    ax.plot(grup['C'], grup['mean'], marker='o', label=NAMA_SKEMA[skema])
    ax.fill_between(grup['C'], grup['mean'] - grup['std'], grup['mean'] + grup['std'], alpha=0.15)

ax.set_xscale('log')
ax.set_xlabel('Nilai C (skala log)')
ax.set_ylabel('Macro F1 (rata-rata 5 lipatan berbasis grup)')
ax.set_title('Cross-validation SVM per Nilai C dan Skema', fontsize=13, fontweight='bold')
ax.grid(linestyle='--', alpha=0.5)
ax.legend()
plt.show()

In [ ]:
baris_terbaik = df_cv.loc[df_cv.groupby('skema')['mean'].idxmax()]
C_TERBAIK = dict(zip(baris_terbaik['skema'], baris_terbaik['C']))

# skema terbaik ditentukan dari cv, bukan dari data uji
skema_terbaik = baris_terbaik.loc[baris_terbaik['mean'].idxmax(), 'skema']

print(baris_terbaik[['skema', 'C', 'mean', 'std']].to_string(index=False))
print(f"\nskema terbaik (cv): {NAMA_SKEMA[skema_terbaik]} dengan C={C_TERBAIK[skema_terbaik]}")

In [ ]:
pred_uji = {}
hasil_test = []

for skema in SKEMA:
    model = bangun_pipeline(skema, C_TERBAIK[skema])
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    pred_uji[skema] = y_pred

    rep = classification_report(y_test, y_pred, labels=LABELS, output_dict=True, zero_division=0)

    print(f"\n=== {NAMA_SKEMA[skema]} (C={C_TERBAIK[skema]}) ===")
    print(classification_report(y_test, y_pred, labels=LABELS, digits=3, zero_division=0))

    cm = confusion_matrix(y_test, y_pred, labels=LABELS)
    ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(cmap='Blues')
    plt.title(f'Confusion Matrix SVM: {NAMA_SKEMA[skema]} (C={C_TERBAIK[skema]})')
    plt.show()

    hasil_test.append({
        'skema': NAMA_SKEMA[skema],
        'C': C_TERBAIK[skema],
        'cv_macro_f1': df_cv.loc[(df_cv['skema'] == skema) & (df_cv['C'] == C_TERBAIK[skema]), 'mean'].iloc[0],
        'test_macro_f1': rep['macro avg']['f1-score'],
        'test_f1_negatif': rep['negatif']['f1-score'],
        'test_recall_negatif': rep['negatif']['recall'],
        'test_accuracy': rep['accuracy'],
    })

df_perbandingan = pd.DataFrame(hasil_test)
print("\nperbandingan skema:")
print(df_perbandingan.round(4).to_string(index=False))

In [ ]:
rng = np.random.default_rng(42)
y_t = y_test.values
n = len(y_t)
indeks_bootstrap = [rng.integers(0, n, n) for _ in range(1000)]

pred_terbaik = pred_uji[skema_terbaik]

for skema in SKEMA:
    if skema == skema_terbaik:
        continue

    selisih_f1 = []
    selisih_f1_neg = []
    for i in indeks_bootstrap:
        f_a = f1_score(y_t[i], pred_terbaik[i], average='macro', labels=LABELS)
        f_b = f1_score(y_t[i], pred_uji[skema][i], average='macro', labels=LABELS)
        n_a = f1_score(y_t[i], pred_terbaik[i], average=None, labels=LABELS)[1]
        n_b = f1_score(y_t[i], pred_uji[skema][i], average=None, labels=LABELS)[1]
        selisih_f1.append(f_a - f_b)
        selisih_f1_neg.append(n_a - n_b)

    selisih_f1 = np.array(selisih_f1)
    selisih_f1_neg = np.array(selisih_f1_neg)

    print(f"{NAMA_SKEMA[skema_terbaik]} vs {NAMA_SKEMA[skema]}:")
    print(f"  selisih macro f1: {selisih_f1.mean():.4f}, "
          f"95% CI [{np.percentile(selisih_f1, 2.5):.4f}, {np.percentile(selisih_f1, 97.5):.4f}]")
    print(f"  selisih f1 negatif: {selisih_f1_neg.mean():.4f}, "
          f"95% CI [{np.percentile(selisih_f1_neg, 2.5):.4f}, {np.percentile(selisih_f1_neg, 97.5):.4f}]\n")

In [ ]:
# model final dilatih sekali pada data latih, dipakai untuk evaluasi dan inference
model_final = bangun_pipeline(skema_terbaik, C_TERBAIK[skema_terbaik]).fit(X_train, y_train)
pred_final = model_final.predict(X_test)

rep_final = classification_report(y_test, pred_final, labels=LABELS, output_dict=True, zero_division=0)
print("model final (dievaluasi dan dipakai inference) pada uji:")
print(f"  macro f1: {rep_final['macro avg']['f1-score']:.4f}")
print(f"  f1 negatif: {rep_final['negatif']['f1-score']:.4f}, recall negatif: {rep_final['negatif']['recall']:.4f}")
print(f"  akurasi: {rep_final['accuracy']:.4f}")

print(f"\njumlah netral pada uji: {(y_test == 'netral').sum()}")

In [ ]:
import os
os.makedirs('models', exist_ok=True)

KELAS = list(model_final.named_steps['svm'].classes_)

FILE_MODEL = 'models/model_svm_terbaik.joblib'
joblib.dump(
    {'model': model_final, 'skema': skema_terbaik, 'C': C_TERBAIK[skema_terbaik], 'kelas': KELAS},
    FILE_MODEL,
)
print(f"model disimpan ke {FILE_MODEL}: skema {NAMA_SKEMA[skema_terbaik]}, C={C_TERBAIK[skema_terbaik]}")

In [ ]:
bundle = muat_model()
print(f"model dimuat: {bundle['skema']}, C={bundle['C']}, kelas {bundle['kelas']}")

kalimat_uji = [
    "aplikasinya bagus sekali, membantu saya belajar mengaji",
    "langganannya mahal dan sering error",
    "oke aja, lumayan untuk dicoba",
    "ingat ya harus langganan untuk aplikasinya",
]

for teks in kalimat_uji:
    hasil = prediksi(teks, bundle)
    if hasil is None:
        print(f"{teks}\n  teks kosong setelah preprocessing\n")
        continue
    print(teks)
    print(f"  label: {hasil['label']} (skor relatif {hasil['skor_relatif']}%)")
    print(f"  semua kelas: {hasil['semua']}")
    print(f"  skor mentah: {hasil['skor_mentah']}\n")